In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor

df = pd.read_csv("../data/processed/m5_model_df_small.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["id", "date"]).reset_index(drop=True)

print("Rows:", len(df), "Unique series:", df["id"].nunique())
print(df["date"].min(), "to", df["date"].max())


Rows: 365500 Unique series: 500
2014-04-25 00:00:00 to 2016-04-24 00:00:00


In [2]:
# Create lags per series
for lag in [7, 14, 28]:
    df[f"lag_{lag}"] = df.groupby("id")["sales"].shift(lag)

# Rolling means per series (use shifted so we don't leak today's sales)
for w in [7, 14, 28]:
    df[f"roll_mean_{w}"] = (
        df.groupby("id")["sales"]
          .shift(1)
          .rolling(window=w)
          .mean()
          .reset_index(level=0, drop=True)
    )

# Date features
df["dayofweek"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)

# Handle events (simple encoding)
df["event_name_1"] = df["event_name_1"].fillna("none")
df["event_type_1"] = df["event_type_1"].fillna("none")

# Drop rows with missing lag features (first 28 days per series)
df_ml = df.dropna(subset=["lag_28", "roll_mean_28"]).copy()
df_ml.shape


(351500, 27)

In [3]:
cat_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "event_name_1", "event_type_1"]
num_cols = ["lag_7", "lag_14", "lag_28", "roll_mean_7", "roll_mean_14", "roll_mean_28", "dayofweek", "month", "weekofyear"]

X = df_ml[cat_cols + num_cols]
y = df_ml["sales"].values

X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=False)
X_enc.shape


(351500, 774)

In [4]:
HORIZON = 28
max_date = df_ml["date"].max()
val_start = max_date - pd.Timedelta(days=HORIZON - 1)

train_idx = df_ml["date"] < val_start
val_idx = df_ml["date"] >= val_start

X_train, y_train = X_enc.loc[train_idx], y[train_idx]
X_val, y_val = X_enc.loc[val_idx], y[val_idx]

print("Train rows:", X_train.shape[0], "Val rows:", X_val.shape[0])
print("Val start:", val_start.date(), "Max date:", max_date.date())


Train rows: 337500 Val rows: 14000
Val start: 2016-03-28 Max date: 2016-04-24


In [5]:
model = HistGradientBoostingRegressor(
    max_depth=8,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

model.fit(X_train, y_train)
pred = model.predict(X_val)
pred = np.clip(pred, 0, None)  # sales can't be negative


In [6]:
def wape(y, yhat):
    denom = np.sum(np.abs(y))
    return np.sum(np.abs(y - yhat)) / denom if denom != 0 else np.nan

mae = mean_absolute_error(y_val, pred)
rmse = np.sqrt(mean_squared_error(y_val, pred))
wape_score = wape(y_val, pred)

print("ML MAE:", mae)
print("ML RMSE:", rmse)
print("ML WAPE:", wape_score)


ML MAE: 3.5942152795798505
ML RMSE: 5.6251892307524844
ML WAPE: 0.4025939810869763


In [7]:
ml_results = {
    "Model": "HistGradientBoostingRegressor (sklearn)",
    "MAE": float(mae),
    "RMSE": float(rmse),
    "WAPE": float(wape_score),
    "Baseline_WAPE_MA28": 0.497139  # from your baseline table
}
ml_results



{'Model': 'HistGradientBoostingRegressor (sklearn)',
 'MAE': 3.5942152795798505,
 'RMSE': 5.6251892307524844,
 'WAPE': 0.4025939810869763,
 'Baseline_WAPE_MA28': 0.497139}